# Importing Libraries

In [16]:
import numpy as np
from pynq import Overlay, allocate
import subprocess
import ctypes
import time
import os

# Loading Overlay

In [17]:
overlay = Overlay("bitstream_files/polyvec_accel.bit") 
accel = overlay.polyvec_accel_0 

# Compiling C code for reference

In [18]:
compile_cmd = [
    "gcc", "-shared", "-fPIC",
    "-Wall", "-Wextra", "-Wpedantic", "-Wmissing-prototypes", 
    "-Wredundant-decls", "-Wshadow", "-Wpointer-arith", 
    "-O3", "-fomit-frame-pointer", "-z", "noexecstack",
    "-Wno-unused-result",
    "polyvec_files/ntt.c", "polyvec_files/poly.c", "polyvec_files/polyvec.c", 
    "polyvec_files/reduce.c", "polyvec_files/verify.c",
    "-o", "kyber_ref_clean.so"
]
subprocess.run(compile_cmd, check=True, capture_output=True)

CompletedProcess(args=['gcc', '-shared', '-fPIC', '-Wall', '-Wextra', '-Wpedantic', '-Wmissing-prototypes', '-Wredundant-decls', '-Wshadow', '-Wpointer-arith', '-O3', '-fomit-frame-pointer', '-z', 'noexecstack', '-Wno-unused-result', 'polyvec_files/ntt.c', 'polyvec_files/poly.c', 'polyvec_files/polyvec.c', 'polyvec_files/reduce.c', 'polyvec_files/verify.c', '-o', 'kyber_ref_clean.so'], returncode=0, stdout=b'', stderr=b'polyvec_files/ntt.c: In function \xe2\x80\x98fqmul\xe2\x80\x99:\npolyvec_files/ntt.c:69: warning: ignoring \xe2\x80\x98#pragma HLS INLINE\xe2\x80\x99 [-Wunknown-pragmas]\n   69 |     #pragma HLS INLINE\n      | \npolyvec_files/ntt.c: In function \xe2\x80\x98ntt\xe2\x80\x99:\npolyvec_files/ntt.c:83: warning: ignoring \xe2\x80\x98#pragma HLS ARRAY_PARTITION\xe2\x80\x99 [-Wunknown-pragmas]\n   83 | #pragma HLS ARRAY_PARTITION variable=r type=cyclic factor=16 dim=1\n      | \npolyvec_files/ntt.c:93: warning: ignoring \xe2\x80\x98#pragma HLS PIPELINE\xe2\x80\x99 [-Wunknown-p

# Loading C library

In [19]:
kyber_ref = ctypes.CDLL("./kyber_ref_clean.so")
ptr_int16 = ctypes.POINTER(ctypes.c_int16)

kyber_ref.pqcrystals_kyber768_ref_polyvec_ntt.argtypes = [ptr_int16]
kyber_ref.pqcrystals_kyber768_ref_polyvec_invntt_tomont.argtypes = [ptr_int16]
kyber_ref.pqcrystals_kyber768_ref_polyvec_basemul_acc_montgomery.argtypes = [ptr_int16, ptr_int16, ptr_int16]
kyber_ref.pqcrystals_kyber768_ref_polyvec_add.argtypes = [ptr_int16, ptr_int16, ptr_int16]
kyber_ref.pqcrystals_kyber768_ref_polyvec_reduce.argtypes = [ptr_int16]

kyber_ref.pqcrystals_kyber768_ref_poly_tomont.argtypes = [ptr_int16]
kyber_ref.pqcrystals_kyber768_ref_poly_invntt_tomont.argtypes = [ptr_int16]
kyber_ref.pqcrystals_kyber768_ref_poly_add.argtypes = [ptr_int16, ptr_int16, ptr_int16]
kyber_ref.pqcrystals_kyber768_ref_poly_sub.argtypes = [ptr_int16, ptr_int16, ptr_int16]
kyber_ref.pqcrystals_kyber768_ref_poly_reduce.argtypes = [ptr_int16]

# Parameters

In [20]:
#Kyber Parameters
KYBER_N = 256
KYBER_K = 3
KYBER_Q = 3329
ITERATIONS = 10000

#Op codes
OP_CODES = {
    0: "OP_POLYVEC_NTT",
    1: "OP_POLYVEC_INVNTT_TOMONT",
    2: "OP_POLYVEC_BASEMUL_ACC_MONTGOMERY",
    3: "OP_POLYVEC_ADD",
    4: "OP_POLYVEC_REDUCE",
    5: "OP_POLY_TOMONT",
    6: "OP_POLY_INVNTT_TOMONT",
    7: "OP_POLY_ADD",
    8: "OP_POLY_SUB",
    9: "OP_POLY_REDUCE"
}

# Auxiliary Functions

In [21]:
def set_pointer(accel, prefix_name, physical_address):
    lower_32 = physical_address & 0xFFFFFFFF
    upper_32 = (physical_address >> 32) & 0xFFFFFFFF
    setattr(accel.register_map, f"{prefix_name}_1", lower_32)
    setattr(accel.register_map, f"{prefix_name}_2", upper_32)

In [22]:
def run_hardware(op):
    accel.write(0x10, op) # write op_code
    accel.write(0x00, 1) # start

    while (accel.read(0x00) & 0x02) == 0: # wait for done
        pass

# Allocating Memory

In [23]:
mem_pv_r = allocate(shape=(KYBER_N*KYBER_K,), dtype=np.int16)
mem_pv_a = allocate(shape=(KYBER_N*KYBER_K,), dtype=np.int16)
mem_pv_b = allocate(shape=(KYBER_N*KYBER_K,), dtype=np.int16)

mem_p_r = allocate(shape=(KYBER_N,), dtype=np.int16)
mem_p_a = allocate(shape=(KYBER_N,), dtype=np.int16)
mem_p_b = allocate(shape=(KYBER_N,), dtype=np.int16)

# Generating random numbers

In [24]:
mem_pv_a[:] = np.random.randint(0, KYBER_Q, size=KYBER_N*KYBER_K, dtype=np.int16)
mem_pv_b[:] = np.random.randint(0, KYBER_Q, size=KYBER_N*KYBER_K, dtype=np.int16)

mem_p_a[:] = np.random.randint(0, KYBER_Q, size=KYBER_N, dtype=np.int16)
mem_p_b[:] = np.random.randint(0, KYBER_Q, size=KYBER_N, dtype=np.int16)

# Mapping Physical Addresses

In [25]:
set_pointer(accel, "mem_pv_r", mem_pv_r.physical_address)
set_pointer(accel, "mem_pv_a", mem_pv_a.physical_address)
set_pointer(accel, "mem_pv_b", mem_pv_b.physical_address)

set_pointer(accel, "mem_p_r", mem_p_r.physical_address)
set_pointer(accel, "mem_p_a", mem_p_a.physical_address)
set_pointer(accel, "mem_p_b", mem_p_b.physical_address)

# Validation Tests

In [26]:
total_errors = 0

for op_code, op_name in OP_CODES.items():
    print(f"Testing {op_name} ({ITERATIONS} iterations)")
    op_errors = 0  
    
    for i in range(ITERATIONS):
        a_pv = np.random.randint(0, KYBER_Q, size=KYBER_N*KYBER_K, dtype=np.int16)
        b_pv = np.random.randint(0, KYBER_Q, size=KYBER_N*KYBER_K, dtype=np.int16)
        a_p  = np.random.randint(0, KYBER_Q, size=KYBER_N, dtype=np.int16)
        b_p  = np.random.randint(0, KYBER_Q, size=KYBER_N, dtype=np.int16)
        
        mem_pv_a[:] = a_pv
        mem_pv_b[:] = b_pv
        mem_p_a[:]  = a_p
        mem_p_b[:]  = b_p
        mem_pv_r[:] = 0 
        mem_p_r[:]  = 0
        
        ref_pv_r = np.zeros(KYBER_N*KYBER_K, dtype=np.int16)
        ref_p_r  = np.zeros(KYBER_N, dtype=np.int16)
        
        c_a_pv = a_pv.ctypes.data_as(ptr_int16)
        c_b_pv = b_pv.ctypes.data_as(ptr_int16)
        c_r_pv = ref_pv_r.ctypes.data_as(ptr_int16)
        
        c_a_p  = a_p.ctypes.data_as(ptr_int16)
        c_b_p  = b_p.ctypes.data_as(ptr_int16)
        c_r_p  = ref_p_r.ctypes.data_as(ptr_int16)
        
        hw_val = None
        sw_val = None
        
        if op_code == 0: 
            ref_pv_r[:] = a_pv; mem_pv_r[:] = a_pv
            kyber_ref.pqcrystals_kyber768_ref_polyvec_ntt(c_r_pv)
            run_hardware(op_code)
            hw_val = mem_pv_r; sw_val = ref_pv_r
            
        elif op_code == 1: 
            ref_pv_r[:] = a_pv; mem_pv_r[:] = a_pv
            kyber_ref.pqcrystals_kyber768_ref_polyvec_invntt_tomont(c_r_pv)
            run_hardware(op_code)
            hw_val = mem_pv_r; sw_val = ref_pv_r
            
        elif op_code == 2: 
            kyber_ref.pqcrystals_kyber768_ref_polyvec_basemul_acc_montgomery(c_r_p, c_a_pv, c_b_pv)
            run_hardware(op_code)
            hw_val = mem_p_r; sw_val = ref_p_r
            
        elif op_code == 3: 
            kyber_ref.pqcrystals_kyber768_ref_polyvec_add(c_r_pv, c_a_pv, c_b_pv)
            run_hardware(op_code)
            hw_val = mem_pv_r; sw_val = ref_pv_r
            
        elif op_code == 4: 
            ref_pv_r[:] = a_pv; mem_pv_r[:] = a_pv
            kyber_ref.pqcrystals_kyber768_ref_polyvec_reduce(c_r_pv)
            run_hardware(op_code)
            hw_val = mem_pv_r; sw_val = ref_pv_r
            
        elif op_code == 5: 
            ref_p_r[:] = a_p; mem_p_r[:] = a_p
            kyber_ref.pqcrystals_kyber768_ref_poly_tomont(c_r_p)
            run_hardware(op_code)
            hw_val = mem_p_r; sw_val = ref_p_r
            
        elif op_code == 6: 
            ref_p_r[:] = a_p; mem_p_r[:] = a_p
            kyber_ref.pqcrystals_kyber768_ref_poly_invntt_tomont(c_r_p)
            run_hardware(op_code)
            hw_val = mem_p_r; sw_val = ref_p_r
            
        elif op_code == 7: 
            kyber_ref.pqcrystals_kyber768_ref_poly_add(c_r_p, c_a_p, c_b_p)
            run_hardware(op_code)
            hw_val = mem_p_r; sw_val = ref_p_r
            
        elif op_code == 8: 
            kyber_ref.pqcrystals_kyber768_ref_poly_sub(c_r_p, c_a_p, c_b_p)
            run_hardware(op_code)
            hw_val = mem_p_r; sw_val = ref_p_r
            
        elif op_code == 9: 
            ref_p_r[:] = a_p; mem_p_r[:] = a_p
            kyber_ref.pqcrystals_kyber768_ref_poly_reduce(c_r_p)
            run_hardware(op_code)
            hw_val = mem_p_r; sw_val = ref_p_r

        diff = hw_val.astype(np.int32) - sw_val.astype(np.int32)
        mod_diff = diff % KYBER_Q
        real_errors = np.sum((mod_diff != 0) & (diff != 0))

        if real_errors > 0:
            op_errors += 1
            total_errors += 1
            
    if op_errors == 0:
        print(f"Test {op_name} passed all {ITERATIONS} iterations.")
    else:
        print(f"Test {op_name} failed in {op_errors} out of {ITERATIONS} iterations.")
    print("-" * 50)
    
if total_errors == 0:
    print("Success! All operations are correct across all iterations.")
else:
    print(f"Failed. Found errors in {total_errors} test iterations overall.")

Testing OP_POLYVEC_NTT (10000 iterations)
Test OP_POLYVEC_NTT passed all 10000 iterations.
--------------------------------------------------
Testing OP_POLYVEC_INVNTT_TOMONT (10000 iterations)
Test OP_POLYVEC_INVNTT_TOMONT passed all 10000 iterations.
--------------------------------------------------
Testing OP_POLYVEC_BASEMUL_ACC_MONTGOMERY (10000 iterations)
Test OP_POLYVEC_BASEMUL_ACC_MONTGOMERY passed all 10000 iterations.
--------------------------------------------------
Testing OP_POLYVEC_ADD (10000 iterations)
Test OP_POLYVEC_ADD passed all 10000 iterations.
--------------------------------------------------
Testing OP_POLYVEC_REDUCE (10000 iterations)
Test OP_POLYVEC_REDUCE passed all 10000 iterations.
--------------------------------------------------
Testing OP_POLY_TOMONT (10000 iterations)
Test OP_POLY_TOMONT passed all 10000 iterations.
--------------------------------------------------
Testing OP_POLY_INVNTT_TOMONT (10000 iterations)
Test OP_POLY_INVNTT_TOMONT passed a

In [27]:
CPU_FREQ_HZ = 650_000_000 
FPGA_FREQ_HZ = 100_000_000 

print(f"Frequency -> CPU: {CPU_FREQ_HZ/1e6:.2f} MHz | FPGA: {FPGA_FREQ_HZ/1e6:.2f} MHz")
print(f"{'Operation':<35} | {'Time SW (us)':<12} | {'Time HW (us)':<12} | {'Cycles SW':<12} | {'Cycles HW':<12} | {'Speedup'}")
print("-" * 110)

for op_code, op_name in OP_CODES.items():
    total_time_sw = 0.0
    total_time_hw = 0.0

    c_a_pv = mem_pv_a.ctypes.data_as(ptr_int16)
    c_b_pv = mem_pv_b.ctypes.data_as(ptr_int16)
    c_r_pv = mem_pv_r.ctypes.data_as(ptr_int16)
    
    c_a_p  = mem_p_a.ctypes.data_as(ptr_int16)
    c_b_p  = mem_p_b.ctypes.data_as(ptr_int16)
    c_r_p  = mem_p_r.ctypes.data_as(ptr_int16)

    for _ in range(ITERATIONS):
        mem_pv_a[:] = np.random.randint(0, KYBER_Q, size=KYBER_N*KYBER_K, dtype=np.int16)
        mem_pv_b[:] = np.random.randint(0, KYBER_Q, size=KYBER_N*KYBER_K, dtype=np.int16)
        mem_p_a[:]  = np.random.randint(0, KYBER_Q, size=KYBER_N, dtype=np.int16)
        mem_p_b[:]  = np.random.randint(0, KYBER_Q, size=KYBER_N, dtype=np.int16)
        
        start_sw = time.perf_counter()
        if op_code == 0:   kyber_ref.pqcrystals_kyber768_ref_polyvec_ntt(c_r_pv)
        elif op_code == 1: kyber_ref.pqcrystals_kyber768_ref_polyvec_invntt_tomont(c_r_pv)
        elif op_code == 2: kyber_ref.pqcrystals_kyber768_ref_polyvec_basemul_acc_montgomery(c_r_p, c_a_pv, c_b_pv)
        elif op_code == 3: kyber_ref.pqcrystals_kyber768_ref_polyvec_add(c_r_pv, c_a_pv, c_b_pv)
        elif op_code == 4: kyber_ref.pqcrystals_kyber768_ref_polyvec_reduce(c_r_pv)
        elif op_code == 5: kyber_ref.pqcrystals_kyber768_ref_poly_tomont(c_r_p)
        elif op_code == 6: kyber_ref.pqcrystals_kyber768_ref_poly_invntt_tomont(c_r_p)
        elif op_code == 7: kyber_ref.pqcrystals_kyber768_ref_poly_add(c_r_p, c_a_p, c_b_p)
        elif op_code == 8: kyber_ref.pqcrystals_kyber768_ref_poly_sub(c_r_p, c_a_p, c_b_p)
        elif op_code == 9: kyber_ref.pqcrystals_kyber768_ref_poly_reduce(c_r_p)
        end_sw = time.perf_counter()
        
        total_time_sw += (end_sw - start_sw)
        
        start_hw = time.perf_counter()
        run_hardware(op_code)
        end_hw = time.perf_counter()
        
        total_time_hw += (end_hw - start_hw)

    avg_sw_s = (total_time_sw / ITERATIONS)
    avg_hw_s = (total_time_hw / ITERATIONS)
    
    avg_sw_us = avg_sw_s * 1_000_000
    avg_hw_us = avg_hw_s * 1_000_000
    
    cycles_sw = avg_sw_s * CPU_FREQ_HZ
    cycles_hw = avg_hw_s * FPGA_FREQ_HZ
    
    speedup = avg_sw_s / avg_hw_s
    
    print(f"{op_name:<35} | {avg_sw_us:<12.2f} | {avg_hw_us:<12.2f} | {int(cycles_sw):<12} | {int(cycles_hw):<12} | {speedup:.2f}x")

print("-" * 110)

Frequency -> CPU: 650.00 MHz | FPGA: 100.00 MHz
Operation                           | Time SW (us) | Time HW (us) | Cycles SW    | Cycles HW    | Speedup
--------------------------------------------------------------------------------------------------------------
OP_POLYVEC_NTT                      | 1101.61      | 236.93       | 716047       | 23693        | 4.65x
OP_POLYVEC_INVNTT_TOMONT            | 1187.71      | 244.35       | 772008       | 24434        | 4.86x
OP_POLYVEC_BASEMUL_ACC_MONTGOMERY   | 427.78       | 121.99       | 278056       | 12199        | 3.51x
OP_POLYVEC_ADD                      | 78.85        | 121.92       | 51250        | 12192        | 0.65x
OP_POLYVEC_REDUCE                   | 192.28       | 121.22       | 124982       | 12122        | 1.59x
OP_POLY_TOMONT                      | 91.46        | 99.73        | 59451        | 9972         | 0.92x
OP_POLY_INVNTT_TOMONT               | 424.18       | 140.72       | 275719       | 14071        | 3.01x
OP_POLY